# Tutorial: Through-Thickness Thermal Buckling with Elastic Boundaries

In this tutorial, we introduce two major structural complexities:
1. **Surface Convection:** Instead of a uniform internal heat source, the panel is heated by a hot fluid on its top surface and cooled by a fluid on its bottom surface. This creates a through-thickness temperature gradient.
2. **Elastic Boundary Conditions:** Real-world mounts are rarely perfectly rigid. We will compare a perfectly clamped panel against one with elastic supports (Robin boundary conditions) that allow a slight amount of movement under extreme stress.

We will run both scenarios, generate animations for each to visualize the deformation differences, and finally overlay their bifurcation points to see how support stiffness affects the critical buckling time.

# Standard imports

In [ ]:
try:
    from firedrake import *
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/firedrake-install-release-real.sh" -O "/tmp/firedrake-install.sh" && bash "/tmp/firedrake-install.sh"
    from firedrake import *

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML
from tqdm.auto import tqdm

# Shared Mesh and Function Spaces

Because we are dealing with a severe through-thickness temperature gradient, we significantly increase the vertical mesh resolution to 20 elements.

In [ ]:
L, H = 10.0, 0.1
mesh = RectangleMesh(100, 20, L, H)

V_T = FunctionSpace(mesh, "CG", 1)
V_u = VectorFunctionSpace(mesh, "CG", 1)
W = V_u * V_T  

coords = mesh.coordinates.dat.data.copy()
d = mesh.topological_dimension()
I = Identity(d)
x, y = SpatialCoordinate(mesh)

# Base Properties and Constants

We define our base material properties, time-stepping variables, and the convection parameters.

In [ ]:
# Time-stepping constants
dt_val = 0.001
dt = Constant(dt_val)
n_steps = 200
time_c = Constant(0.0) # Optimized time tracker

# Material properties
T_ref = Constant(300.0) 
rho_c = Constant(1.0) 
k = Constant(0.01) 
nu = Constant(0.3)
E_0 = Constant(1e6)
alpha_0 = Constant(2e-4)

# Convection Parameters
h_top = Constant(50.0)
h_bot = Constant(10.0)
T_env_bot = Constant(300.0)

# Initial geometric imperfection
imperfection = as_vector([0.0, 1e-2 * sin(pi * x / L)])

# Part 1: Fixed Boundary Conditions

## Mathematical Derivation

We utilize the same fully nonlinear material and geometric formulations as the previous tutorial. However, the thermal weak form is modified to replace volumetric heating with surface convection.

### Convection Weak Form
The heat flux across the top and bottom boundaries is modeled using Newton's Law of Cooling:
$$\int_{\partial \Omega_{top}} h_{top} (T - T_{env\_top}) v_T \, ds + \int_{\partial \Omega_{bot}} h_{bot} (T - T_{env\_bot}) v_T \, ds$$

In Firedrake, `ds(4)` refers to the top boundary and `ds(3)` refers to the bottom boundary of a standard `RectangleMesh`.

For this first run, we apply absolute rigid constraints (Dirichlet) to both the left and right edges.

In [ ]:
w1 = Function(W, name="State_1")
w1_old = Function(W, name="State_old_1")
u1, T1 = split(w1)
u1_old, T1_old = split(w1_old)

w1.sub(1).interpolate(T_ref)
w1_old.sub(1).interpolate(T_ref)
w1.sub(0).project(imperfection)
w1_old.sub(0).project(imperfection)

# Boundary Conditions (Rigid Dirichlet)
bc_u_left = DirichletBC(W.sub(0), Constant((0.0, 0.0)), 1)
bc_u_right = DirichletBC(W.sub(0), Constant((0.0, 0.0)), 2)
bcs_fixed = [bc_u_left, bc_u_right]

# Temperature-Dependent Material Properties
E_T1 = E_0 * exp(-0.005 * (T1 - T_ref)) 
alpha_T1 = alpha_0 * (1.0 + 0.02 * (T1 - T_ref))
lmbda_T1 = (E_T1 * nu) / ((1 + nu) * (1 - 2 * nu))
mu_T1 = E_T1 / (2 * (1 + nu))
beta_T1 = alpha_T1 * (3 * lmbda_T1 + 2 * mu_T1)

# Kinematics & Stresses
F_def1 = I + grad(u1)
C1 = F_def1.T * F_def1
E_GL1 = 0.5 * (C1 - I)
S1 = lmbda_T1 * tr(E_GL1) * I + 2 * mu_T1 * E_GL1 - beta_T1 * (T1 - T_ref) * I
P1 = F_def1 * S1

# Variational Problem
v_u1, v_T1 = TestFunctions(W)
F_u1 = inner(P1, grad(v_u1)) * dx

# Convection Thermal Formulation
T1_rate = (T1 - T1_old) / dt
T_env_top_expr = T_ref + 500.0 * time_c # Optimized time-dependent environment temperature

F_T1_bulk = (rho_c * T1_rate * v_T1 + inner(k * grad(T1), grad(v_T1))) * dx
F_T1_conv = h_top * (T1 - T_env_top_expr) * v_T1 * ds(4) + h_bot * (T1 - T_env_bot) * v_T1 * ds(3)

prob1 = NonlinearVariationalProblem(F_u1 + F_T1_bulk + F_T1_conv, w1, bcs=bcs_fixed)
solver1 = NonlinearVariationalSolver(prob1)

# Part 1: Solver Loop and Animation

In [ ]:
# Animation Setup
fig1, ax1 = plt.subplots(1, 1, figsize=(10, 3))
ax1.set_title("Through-Thickness Buckling (Fixed Boundaries)")
ax1.set_xlabel("x [m]")
ax1.set_ylabel("y [m]")
ax1.set_xlim(-0.5, L + 0.5)
ax1.set_ylim(-2.0, 2.0) 
ax1.set_aspect('equal') 
frames1 = []

time_list_1 = []
deflection_list_1 = []
t = 0.0
time_c.assign(t)

# Expected max temp: 300 + (500 * 200 * 0.001) = 400.0
T_max_expected = 400.0 

for step in tqdm(range(n_steps), desc="Solving Fixed Boundaries"):
    t += dt_val
    time_c.assign(t)
    
    solver1.solve()
    
    u_out_1, T_out_1 = w1.subfunctions
    abs_max_def = np.max(np.abs(u_out_1.dat.data[:, 1]))
    
    time_list_1.append(t)
    deflection_list_1.append(max(abs_max_def, 1e-10))
    
    # Deform mesh for animation
    mesh.coordinates.dat.data[:] = coords + u_out_1.dat.data[:]
    c1 = tripcolor(T_out_1, axes=ax1, cmap='inferno', vmin=float(T_ref), vmax=T_max_expected)
    frames1.append([c1])
    
    # Reset mesh
    mesh.coordinates.dat.data[:] = coords
    w1_old.assign(w1)

plt.close(fig1)
ani1 = ArtistAnimation(fig1, frames1, interval=50, blit=True, repeat_delay=1000)
display(HTML(f'<div style="width:100%;">{ani1.to_jshtml()}</div>'))

# Part 2: Elastic Boundary Conditions (Robin)

## Mathematical Derivation

To model elastic supports, we remove the explicit Dirichlet boundary conditions (`bcs = []`). Instead, we add a spring-like penalty term to the variational form. 

The energy required to deform the boundaries is proportional to a stiffness constant $K_{elastic}$:
$$\int_{\partial \Omega_{left}} K_{elastic} u \cdot v_u \, ds + \int_{\partial \Omega_{right}} K_{elastic} u \cdot v_u \, ds$$

In Firedrake, `ds(1)` is the left boundary and `ds(2)` is the right boundary. A very high $K_{elastic}$ approaches a rigid clamp, while a lower value allows the panel ends to compress slightly inwards during heating, relieving some of the compressive stress.

In [ ]:
w2 = Function(W, name="State_2")
w2_old = Function(W, name="State_old_2")
u2, T2 = split(w2)
u2_old, T2_old = split(w2_old)

w2.sub(1).interpolate(T_ref)
w2_old.sub(1).interpolate(T_ref)
w2.sub(0).project(imperfection)
w2_old.sub(0).project(imperfection)

# Elastic Boundary Stiffness (1e12 = highly stiff but not perfectly rigid)
K_elastic = Constant(1e8) 

# Temperature-Dependent Material Properties
E_T2 = E_0 * exp(-0.005 * (T2 - T_ref)) 
alpha_T2 = alpha_0 * (1.0 + 0.02 * (T2 - T_ref))
lmbda_T2 = (E_T2 * nu) / ((1 + nu) * (1 - 2 * nu))
mu_T2 = E_T2 / (2 * (1 + nu))
beta_T2 = alpha_T2 * (3 * lmbda_T2 + 2 * mu_T2)

# Kinematics & Stresses
F_def2 = I + grad(u2)
C2 = F_def2.T * F_def2
E_GL2 = 0.5 * (C2 - I)
S2 = lmbda_T2 * tr(E_GL2) * I + 2 * mu_T2 * E_GL2 - beta_T2 * (T2 - T_ref) * I
P2 = F_def2 * S2

# Variational Problem
v_u2, v_T2 = TestFunctions(W)

# Internal Mechanical Work
F_u2_internal = inner(P2, grad(v_u2)) * dx

# Robin Boundary Term for Elasticity
F_u2_elastic = inner(K_elastic * u2, v_u2) * ds(1) + inner(K_elastic * u2, v_u2) * ds(2)
F_u2_total = F_u2_internal + F_u2_elastic

# Thermal Problem
T2_rate = (T2 - T2_old) / dt
F_T2_bulk = (rho_c * T2_rate * v_T2 + inner(k * grad(T2), grad(v_T2))) * dx
F_T2_conv = h_top * (T2 - T_env_top_expr) * v_T2 * ds(4) + h_bot * (T2 - T_env_bot) * v_T2 * ds(3)

# Notice bcs=[] because constraints are handled via F_u2_elastic
prob2 = NonlinearVariationalProblem(F_u2_total + F_T2_bulk + F_T2_conv, w2, bcs=[])
solver2 = NonlinearVariationalSolver(prob2)

# Part 2: Solver Loop and Animation

In [ ]:
# Animation Setup
fig2, ax2 = plt.subplots(1, 1, figsize=(10, 3))
ax2.set_title("Through-Thickness Buckling (Elastic Boundaries)")
ax2.set_xlabel("x [m]")
ax2.set_ylabel("y [m]")
ax2.set_xlim(-0.5, L + 0.5)
ax2.set_ylim(-2.0, 2.0) 
ax2.set_aspect('equal') 
frames2 = []

time_list_2 = []
deflection_list_2 = []

t = 0.0
time_c.assign(t)

for step in tqdm(range(n_steps), desc="Solving Elastic Boundaries"):
    t += dt_val
    time_c.assign(t)
    
    solver2.solve()
    
    u_out_2, T_out_2 = w2.subfunctions
    abs_max_def_2 = np.max(np.abs(u_out_2.dat.data[:, 1]))
    
    time_list_2.append(t)
    deflection_list_2.append(max(abs_max_def_2, 1e-10))
    
    # Deform mesh for animation
    mesh.coordinates.dat.data[:] = coords + u_out_2.dat.data[:]
    c2 = tripcolor(T_out_2, axes=ax2, cmap='inferno', vmin=float(T_ref), vmax=T_max_expected)
    frames2.append([c2])
    
    # Reset mesh
    mesh.coordinates.dat.data[:] = coords
    w2_old.assign(w2)

plt.close(fig2)
ani2 = ArtistAnimation(fig2, frames2, interval=50, blit=True, repeat_delay=1000)
display(HTML(f'<div style="width:100%;">{ani2.to_jshtml()}</div>'))

# Overlapping Bifurcation Analysis

By plotting both conditions together, you will see how boundary rigidity affects the structure's critical limits. The elastic boundary condition provides slight relief from the thermal expansion, often delaying the onset of buckling compared to the strictly clamped boundaries.

In [ ]:
plt.figure(figsize=(10, 6))

plt.semilogy(time_list_1, deflection_list_1, 'r-', linewidth=2, label='Fixed (Rigid Dirichlet)')
plt.semilogy(time_list_2, deflection_list_2, 'b--', linewidth=2, label='Elastic (Robin penalty)')

plt.xlabel('Time (s)')
plt.ylabel('Max Absolute Deflection (m) [Log Scale]')
plt.title('Bifurcation Comparison: Fixed vs. Elastic Boundary Constraints')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.show()